# 05 - Register the model

MLflow Model Registry is the source of truth. SageMaker Model Registry is a
mirror.

### How the mirror works

The tracking server was created with `AutomaticModelRegistration=True`. When you
register a version in MLflow, SageMaker mirrors it into the SageMaker Model
Registry as a model package, in a group named after the MLflow registered model.

That gives you one place to look (MLflow) while keeping the SageMaker deploy
path, Studio's registry UI and anything else reading model packages working.

Two caveats the code handles:
- the mirror is **asynchronous** (`wait_for_mirrored_package` polls);
- it can be **absent**, so notebook 06's deploy has a documented fallback that
  builds the SageMaker model straight from the training job's artifact.

### Stage = alias, not stage

MLflow's `stages` API is deprecated. This project uses **aliases**:
`@candidate` on registration, then `@staging`, then `@production`.
An alias is a movable pointer, so promotion never copies a model.

In [1]:
import json
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Put this kernel's venv bin/ first on PATH so shell commands (dvc, ...) resolve to the
# venv, however Jupyter was launched.
import sys
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params)
print("region:", settings.region, "| bucket:", settings.bucket)

region: us-east-1 | bucket: sagemaker-us-east-1-063676201336


In [2]:
evaluation = json.loads(Path("reports/evaluation.json").read_text())
print(f"run      : {evaluation['run_id']}")
print(f"test AUC : {evaluation['metrics']['test_auc']:.4f}")
print(f"gate     : {'PASS' if evaluation['passed'] else 'FAIL'}")

run      : 0eb9b0c6f641470abae33de9b38e890a
test AUC : 0.9671
gate     : PASS


## 1. Register as `@candidate`

`register()` also copies the lineage tags onto the **model version**, so you can
answer "what data is production trained on?" without walking back to the run.

In [3]:
from churn.registry import ALIAS_CANDIDATE, register

if not evaluation["passed"]:
    raise SystemExit("model missed the AUC gate; not registering")

model = register(
    run_id=evaluation["run_id"],
    settings=settings,
    params=params,
    alias=ALIAS_CANDIDATE,
    description=f"test AUC {evaluation['metrics']['test_auc']:.4f}",
)

print("name    :", model.name)
print("version :", model.version)
print("uri     :", model.model_uri)
print("SM pkg  :", model.sagemaker_model_package_arn or "(mirror not yet visible)")

/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sagemaker.config INFO - Not applying SDK defaults from location: /usr/local/share/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/ritabanchowdhury/Library/Application Support/sagemaker/config.yaml


/Users/ritabanchowdhury/.venvs/churn-mlops/lib/python3.11/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
Registered model 'player-churn-xgboost' already exists. Creating a new version of this model...
2026/10/28 12:02:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 0eb9b0c6f641470abae33de9b38e890a has no artifacts at artifact path 'model', registering model based on models:/m-

  (no SageMaker model package mirrored for player-churn-xgboost v2 after 300s; deployment will fall back to the MLflow artifact)
name    : player-churn-xgboost
version : 2
uri     : models:/player-churn-xgboost/2
SM pkg  : (mirror not yet visible)


In [4]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
client = MlflowClient()

version = client.get_model_version(model.name, model.version)
print("tags on the model version:")
for key, value in sorted(version.tags.items()):
    print(f"  {key:30s} {value}")

tags on the model version:
  dvc.data.interim.features.parquet d0605019cf55f50aaeee3c993fe9847a
  dvc.data.processed             5f68eff31363915f49ff8091544bd6e0.dir
  dvc.reports.split_summary.json bf9ed4c83e081dac839a58512490eb6d
  dvc_pipeline_stages            features split
  gate_passed                    1.0
  git_commit                     694fa803f80447ae330dfb1cbf5e9b9d8a9097d9-dirty
  sagemaker.model_package_arn    arn:aws:sagemaker:us-east-1:063676201336:model-package/player-churn-xgboost-1be1e3/2
  test_auc                       0.9671428571428571


## 2. Confirm the SageMaker mirror

If this comes back empty, give it a minute and re-run. A persistent miss is
survivable — notebook 06 falls back to the MLflow artifact — but worth noticing,
because it usually means `AutomaticModelRegistration` is off on the server.

In [5]:
import boto3

sm = boto3.client("sagemaker", region_name=settings.region)
try:
    packages = sm.list_model_packages(
        ModelPackageGroupName=model.name, SortBy="CreationTime", SortOrder="Descending"
    )["ModelPackageSummaryList"]
    for pkg in packages[:5]:
        print(f"  v{pkg['ModelPackageVersion']:<3} {pkg['ModelApprovalStatus']:<22} {pkg['ModelPackageArn']}")
except sm.exceptions.ClientError as exc:
    print("no mirrored package group yet:", exc.response["Error"]["Message"])

## 3. Promote to `@staging`

Promotion is moving an alias. Nothing is rebuilt or copied.

In [6]:
from churn.registry import ALIAS_STAGING, promote

staged = promote(version=model.version, alias=ALIAS_STAGING, settings=settings, params=params)
print(f"{staged.name} v{staged.version} is now @{ALIAS_STAGING}")

player-churn-xgboost v2 is now @staging


In [7]:
# What each alias points at right now.
for alias in ("candidate", "staging", "production"):
    try:
        version = client.get_model_version_by_alias(model.name, alias)
        auc = version.tags.get("test_auc", "?")
        print(f"  @{alias:<11} -> v{version.version:<4} (test_auc {auc})")
    except Exception:
        print(f"  @{alias:<11} -> unset")

  @candidate   -> v2    (test_auc 0.9671428571428571)
  @staging     -> v2    (test_auc 0.9671428571428571)
  @production  -> unset


In [8]:
Path("reports/registered_model.json").write_text(
    json.dumps(
        {
            "name": model.name,
            "version": model.version,
            "run_id": model.run_id,
            "model_uri": model.model_uri,
            "sagemaker_model_package_arn": model.sagemaker_model_package_arn,
        },
        indent=2,
    ),
    encoding="utf-8",
)
print("CLI equivalent (what CI runs on merge to main):")
print("python -m churn.registry --require-gate --alias candidate")

CLI equivalent (what CI runs on merge to main):
python -m churn.registry --require-gate --alias candidate


---
## Next

**`06-deploy.ipynb`** -- turn `@staging` into an endpoint.

### What changed vs. the original workshop
Old notebook 05 built a `ModelMetrics` object with four `None` URIs, generated a
Model Card, and called `model.register(..., approval_status="PendingManualApproval")`.
The manual approval was the deploy gate.

Now the gate is a **GitHub Environment** with required reviewers (notebook 07),
which is auditable in the same place as the code review. The SageMaker package is
still flipped to `Approved` when `@production` moves, so anything reading the
SageMaker registry stays consistent.